# Create `dim_date`

In [28]:
import holidays
import pandas as pd
from dotenv import load_dotenv

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

In [3]:
def generate_dim_datetime(start: str, end: str) -> pd.DataFrame:
    hours = pd.date_range(start=start, end=end, freq="h")

    # Generate US holidays cho toàn bộ range năm
    start_year = pd.Timestamp(start).year
    end_year   = pd.Timestamp(end).year

    us_holidays = holidays.US(years=range(start_year, end_year + 1), state="NY")
    # NYC Taxi → dùng state="NY" để có thêm holiday của New York

    holiday_dates  = set(us_holidays.keys())
    holiday_names  = {date: name for date, name in us_holidays.items()}

    df = pd.DataFrame({
        "datetime_key":  hours.strftime("%Y%m%d%H").astype(int),
        "full_datetime": hours,
        "hour":          hours.hour,
        "day":           hours.day,
        "day_of_week":   hours.dayofweek,
        "day_name":      hours.day_name(),
        "month":         hours.month,
        "month_name":    hours.month_name(),
        "quarter":       hours.quarter,
        "year":          hours.year,
        "is_weekend":    hours.dayofweek >= 5,
        "is_rush_hour":  hours.hour.isin(range(7, 10)) | hours.hour.isin(range(17, 20)),

        # Holiday columns
        "is_holiday":    hours.normalize().map(lambda d: d.date() in holiday_dates),
        "holiday_name":  hours.normalize().map(lambda d: holiday_names.get(d.date(), None)),
    })

    return df

df = generate_dim_datetime(start='2009-01-01 00:00:00+00', end='2025-12-31 23:59:59+00')

In [27]:
import sys

import psycopg2.extras

sys.path.append("../../../../src")

from utils.db_connection import PostgresConnection


def write_to_postgres(df: pd.DataFrame, table_name: str = "dim_date") -> None:
    """Ghi DataFrame vào bảng PostgreSQL (mode: truncate then insert)."""
    cols = ", ".join(df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))
    insert_sql = f"INSERT INTO public.{table_name} ({cols}) VALUES ({placeholders})"

    values = [tuple(row) for _, row in df.iterrows()]

    with PostgresConnection(
        host="localhost",
        dbname="gold",
    ) as conn, conn.cursor() as cur:
        cur.execute("TRUNCATE TABLE public.dim_date CASCADE")
        psycopg2.extras.execute_batch(cur, insert_sql, values, page_size=5000)
        conn.commit()

    print(f"Inserted {len(df)} rows into public.{table_name}")


write_to_postgres(df)

Inserted 149016 rows into public.dim_date


In [29]:
need_refresh = False
with PostgresConnection(host="localhost", dbname="gold") as conn, conn.cursor() as cur:
    cur.execute(
        "SELECT MIN(full_datetime)::text, MAX(full_datetime)::text FROM public.dim_date"
    )
    row = cur.fetchone()

In [30]:
row

('2009-01-01 00:00:00', '2025-12-31 23:00:00')

In [20]:
df[~(df['holiday_name'].isnull())]

,datetime_key,full_datetime,hour,day,day_of_week,day_name,month,month_name,quarter,year,is_weekend,is_rush_hour,is_holiday,holiday_name
0,2009010100,2009-01-01 00:00:00+00:00,0,1,3,Thursday,1,January,1,2009,False,False,True,New Year's Day
1,2009010101,2009-01-01 01:00:00+00:00,1,1,3,Thursday,1,January,1,2009,False,False,True,New Year's Day
2,2009010102,2009-01-01 02:00:00+00:00,2,1,3,Thursday,1,January,1,2009,False,False,True,New Year's Day
3,2009010103,2009-01-01 03:00:00+00:00,3,1,3,Thursday,1,January,1,2009,False,False,True,New Year's Day
4,2009010104,2009-01-01 04:00:00+00:00,4,1,3,Thursday,1,January,1,2009,False,False,True,New Year's Day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148867,2025122519,2025-12-25 19:00:00+00:00,19,25,3,Thursday,12,December,4,2025,False,True,True,Christmas Day
148868,2025122520,2025-12-25 20:00:00+00:00,20,25,3,Thursday,12,December,4,2025,False,False,True,Christmas Day
148869,2025122521,2025-12-25 21:00:00+00:00,21,25,3,Thursday,12,December,4,2025,False,False,True,Christmas Day
148870,2025122522,2025-12-25 22:00:00+00:00,22,25,3,Thursday,12,December,4,2025,False,False,True,Christmas Day


In [19]:
df[~(df['holiday_name'].isnull())]['holiday_name'].value_counts()

holiday_name
New Year's Day                                     408
Martin Luther King Jr. Day                         408
Lincoln's Birthday                                 408
Independence Day                                   408
Memorial Day                                       408
Thanksgiving Day                                   408
Veterans Day                                       408
Labor Day                                          408
Columbus Day                                       408
Christmas Day                                      408
Susan B. Anthony Day                               336
Washington's Birthday                              336
Election Day                                       336
Independence Day (observed)                        120
Christmas Day (observed)                           120
New Year's Day (observed)                          120
Lincoln's Birthday (observed)                      120
Juneteenth National Independence Day               1

In [14]:
df['holiday_name'][149011]==None

True